# Caption Model Development Evaluation

Evaluate the baseline caption model selected by validation loss on the fixed 200-image development split.

Evaluation covers generated-caption quality and sensitivity to supplying incorrect image features. The checkpoint remains fixed throughout. The final test set remains unused.

## Checkpoint and Evaluation Inputs

Load the saved checkpoint and verify its vocabulary, dataset split, and image preprocessing checksums. Verify the development feature cache before generating captions.

In [1]:
import hashlib
import json
from pathlib import Path

import numpy as np
import torch

project_root = Path.cwd().resolve()
manifest_dir = project_root / "manifests"
cache_dir = project_root / "data" / "processed" / "visual_features"
checkpoint_path = (
    project_root / "data" / "processed" / "caption_training"
    / "baseline_v1" / "best.pt"
)

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)


def file_sha256(path):
    """Calculate a file's SHA-256 checksum using bounded memory.

    Parameters
    ----------
    path : pathlib.Path
        File to read.

    Returns
    -------
    str
        Hexadecimal SHA-256 digest.
    """
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


# Load the locally created checkpoint without executing training cells.
selected_checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True,
)

artifact_checks = {
    "splits_v1.json": selected_checkpoint["split_sha256"],
    "caption_vocabulary_v1.json": selected_checkpoint["vocabulary_sha256"],
    "image_preprocessing_v1.json": (
        selected_checkpoint["image_preprocessing_sha256"]
    ),
}
for filename, expected_checksum in artifact_checks.items():
    if file_sha256(manifest_dir / filename) != expected_checksum:
        raise ValueError(f"Checkpoint input checksum differs: {filename}")

splits = json.loads(
    (manifest_dir / "splits_v1.json").read_text(encoding="utf-8")
)
vocabulary_document = json.loads(
    (manifest_dir / "caption_vocabulary_v1.json").read_text(
        encoding="utf-8"
    )
)
vocabulary_tokens = vocabulary_document["tokens"]
token_to_id = {
    token: index for index, token in enumerate(vocabulary_tokens)
}
id_to_token = dict(enumerate(vocabulary_tokens))

development_metadata = json.loads(
    (cache_dir / "development_resnet18_v1.json").read_text(
        encoding="utf-8"
    )
)
development_image_ids = sorted(splits["image_ids"]["development"])

expected_cache_values = {
    "split": "development",
    "split_manifest_sha256": selected_checkpoint["split_sha256"],
    "image_preprocessing_sha256": (
        selected_checkpoint["image_preprocessing_sha256"]
    ),
    "encoder": "resnet18",
    "encoder_weights": "IMAGENET1K_V1",
    "image_ids": development_image_ids,
    "shape": [len(development_image_ids), 49, 512],
    "dtype": "float32",
}
for key, expected_value in expected_cache_values.items():
    if development_metadata.get(key) != expected_value:
        raise ValueError(f"Development cache metadata differs: {key}")

development_feature_path = (
    cache_dir / development_metadata["feature_file"]
)
if (
    file_sha256(development_feature_path)
    != development_metadata["feature_sha256"]
):
    raise ValueError("Development feature checksum differs.")

development_features = np.load(
    development_feature_path,
    mmap_mode="r",
    allow_pickle=False,
)
if (
    list(development_features.shape) != development_metadata["shape"]
    or development_features.dtype != np.float32
):
    raise ValueError("Development feature shape or dtype differs.")

model_config = selected_checkpoint["model_config"]
if (
    model_config["vocabulary_size"] != len(vocabulary_tokens)
    or model_config["pad_id"] != token_to_id["<pad>"]
):
    raise ValueError("Model configuration and vocabulary differ.")

print("Project root:", project_root)
print("Device:", device)
print("Selected epoch:", selected_checkpoint["epoch"])
print(
    "Selected validation loss:",
    round(selected_checkpoint["best_validation_loss"], 4),
)
print("Vocabulary size:", len(vocabulary_tokens))
print("Development images:", len(development_image_ids))
print("Feature shape:", development_features.shape)
print("Checkpoint input checksums: verified")
print("Development cache: verified")

Project root: /Users/ashlabs/Projects/Masters in AI/github/image-inspired-story-creator
Device: mps
Selected epoch: 7
Selected validation loss: 2.6082
Vocabulary size: 7257
Development images: 200
Feature shape: (200, 49, 512)
Checkpoint input checksums: verified
Development cache: verified


## Load the Trained Decoder

The existing CaptionDecoder class is extracted into a reusable Python module. Only its class definition is copied; training cells are not executed.

The decoder is reconstructed from the checkpoint configuration, then its saved parameters are loaded with strict matching. Evaluation mode disables dropout.

In [2]:
import ast
import importlib

# Read the existing implementation without executing the notebook.
training_notebook = json.loads(
    (project_root / "generative_model.ipynb").read_text(
        encoding="utf-8"
    )
)

decoder_definitions = []
for cell in training_notebook["cells"]:
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])
    parsed = ast.parse(source)

    for node in parsed.body:
        if isinstance(node, ast.ClassDef) and node.name == "CaptionDecoder":
            decoder_definitions.append(
                ast.get_source_segment(source, node)
            )

if len(decoder_definitions) != 1:
    raise ValueError("Expected exactly one CaptionDecoder definition.")

module_source = (
    '"""Transformer decoder for image-conditioned caption generation."""\n\n'
    "import math\n\n"
    "import torch\n"
    "from torch import nn\n\n\n"
    + decoder_definitions[0]
    + "\n"
)

# Check syntax before writing the reusable module.
compile(module_source, "caption_model.py", "exec")

module_path = project_root / "caption_model.py"
if module_path.exists():
    if module_path.read_text(encoding="utf-8") != module_source:
        raise FileExistsError(
            "Existing caption_model.py differs; review before replacing."
        )
else:
    module_path.write_text(module_source, encoding="utf-8")

importlib.invalidate_caches()
from caption_model import CaptionDecoder

selected_model = CaptionDecoder(**model_config).to(device)
selected_model.load_state_dict(
    selected_checkpoint["model_state_dict"],
    strict=True,
)
selected_model.eval()

print("Module:", module_path.name)
print("Checkpoint parameters: loaded with strict matching")
print("Training mode:", selected_model.training)
print(
    "Model parameters:",
    sum(parameter.numel() for parameter in selected_model.parameters()),
)
print("Device:", next(selected_model.parameters()).device)

Module: caption_model.py
Checkpoint parameters: loaded with strict matching
Training mode: False
Model parameters: 5984089
Device: mps:0


## Correct-Image and Mismatched-Image Generation

The selected checkpoint generates captions for all 200 development images using greedy decoding, with at most 39 tokens after the beginning token.

A control condition assigns each image another development image's features. The mapping is fixed using seed 42 and contains no self-matches. Model weights and decoding settings remain identical across conditions.

Caption differences measure sensitivity to image features. Grounding quality requires comparison against the images and reference captions.

In [3]:
def generate_caption_ids(model, features, special_ids, max_new_tokens=39):
    """Generate greedy captions for a batch of image features.

    Parameters
    ----------
    model : CaptionDecoder
        Loaded caption decoder on the same device as features.
    features : torch.Tensor
        Visual features of shape (B, 49, 512).
    special_ids : dict
        IDs for <bos>, <eos>, and <pad>.
    max_new_tokens : int
        Maximum generated tokens after BOS, within model capacity.

    Returns
    -------
    list[list[int]]
        CPU token IDs including BOS and any generated EOS.
        Finished sequences may contain trailing padding.
    """
    if not 1 <= max_new_tokens <= model.text_positions.num_embeddings:
        raise ValueError("Generation length exceeds model capacity.")

    model.eval()
    generated = torch.full(
        (features.shape[0], 1),
        special_ids["<bos>"],
        dtype=torch.long,
        device=features.device,
    )
    finished = torch.zeros(
        features.shape[0], dtype=torch.bool, device=features.device
    )

    with torch.no_grad():
        for _ in range(max_new_tokens):
            length = generated.shape[1]
            causal = torch.triu(
                torch.ones(
                    length, length,
                    dtype=torch.bool,
                    device=features.device,
                ),
                diagonal=1,
            )
            scores = model(
                features,
                generated,
                generated.eq(special_ids["<pad>"]),
                causal,
            )
            next_ids = scores[:, -1].argmax(dim=-1)
            next_ids = next_ids.masked_fill(
                finished, special_ids["<pad>"]
            )
            generated = torch.cat(
                (generated, next_ids.unsqueeze(1)), dim=1
            )
            finished |= next_ids.eq(special_ids["<eos>"])

            if finished.all().item():
                break

    return generated.cpu().tolist()


def readable_caption(token_ids):
    """Convert generated IDs to space-separated text up to EOS.

    Parameters
    ----------
    token_ids : list[int]
        Generated sequence, including BOS.

    Returns
    -------
    str
        Caption with BOS and padding removed; unknown tokens retained.
    """
    words = []
    for token_id in token_ids:
        if token_id == token_to_id["<eos>"]:
            break
        if token_id not in (
            token_to_id["<bos>"], token_to_id["<pad>"]
        ):
            words.append(id_to_token[token_id])
    return " ".join(words)


special_ids = vocabulary_document["special_token_ids"]
image_count = len(development_image_ids)

# Form a cycle through shuffled rows: every row receives another image.
cycle_order = np.random.default_rng(42).permutation(image_count)
mismatched_rows = np.empty(image_count, dtype=np.int64)
mismatched_rows[cycle_order] = np.roll(cycle_order, -1)

assert np.all(mismatched_rows != np.arange(image_count))

development_predictions = []
generation_batch_size = 32

for start in range(0, image_count, generation_batch_size):
    end = min(start + generation_batch_size, image_count)

    correct_features = torch.from_numpy(
        np.array(development_features[start:end], copy=True)
    ).to(device)
    wrong_features = torch.from_numpy(
        np.array(
            development_features[mismatched_rows[start:end]],
            copy=True,
        )
    ).to(device)

    correct_ids = generate_caption_ids(
        selected_model, correct_features, special_ids
    )
    wrong_ids = generate_caption_ids(
        selected_model, wrong_features, special_ids
    )

    for offset, (correct_row, wrong_row) in enumerate(
        zip(correct_ids, wrong_ids)
    ):
        row = start + offset
        development_predictions.append({
            "image_id": development_image_ids[row],
            "mismatched_feature_image_id": development_image_ids[
                int(mismatched_rows[row])
            ],
            "correct_feature_token_ids": correct_row,
            "mismatched_feature_token_ids": wrong_row,
            "correct_feature_caption": readable_caption(correct_row),
            "mismatched_feature_caption": readable_caption(wrong_row),
        })

    del correct_features, wrong_features
    print(f"Generated paired captions: {end}/{image_count}")

changed_captions = sum(
    record["correct_feature_caption"]
    != record["mismatched_feature_caption"]
    for record in development_predictions
)
correct_eos = sum(
    token_to_id["<eos>"] in record["correct_feature_token_ids"]
    for record in development_predictions
)
wrong_eos = sum(
    token_to_id["<eos>"] in record["mismatched_feature_token_ids"]
    for record in development_predictions
)

print(f"\nCaption changes under mismatch: {changed_captions}/{image_count}")
print(f"EOS with correct features: {correct_eos}/{image_count}")
print(f"EOS with mismatched features: {wrong_eos}/{image_count}")

for record in development_predictions[:6]:
    print(f"\nImage ID: {record['image_id']}")
    print("Correct features:", record["correct_feature_caption"])
    print(
        f"Mismatched features "
        f"(image {record['mismatched_feature_image_id']}):",
        record["mismatched_feature_caption"],
    )

Generated paired captions: 32/200
Generated paired captions: 64/200
Generated paired captions: 96/200
Generated paired captions: 128/200
Generated paired captions: 160/200
Generated paired captions: 192/200
Generated paired captions: 200/200

Caption changes under mismatch: 200/200
EOS with correct features: 198/200
EOS with mismatched features: 198/200

Image ID: 1323
Correct features: a couple of giraffes are standing in a zoo enclosure .
Mismatched features (image 581704): a black dog sitting on a couch with a cat on the floor .

Image ID: 2471
Correct features: a cat laying on top of a table with a cat on it .
Mismatched features (image 134072): a bowl of bananas and a banana on a table .

Image ID: 7934
Correct features: a close up of a green apple and a knife
Mismatched features (image 272882): a living room with a couch , chairs , a table , a table , chairs , and a table , a table , chairs , and a table .

Image ID: 8218
Correct features: a large clock tower with a clock on it
M

### Image-Feature Sensitivity Results

All 200 development captions changed when image features were replaced using the fixed mismatch mapping. Both conditions generated an end token for 198 of 200 images.

The model is sensitive to image features. This result does not establish that the generated captions accurately describe those images. The displayed examples still contain incorrect objects and unsupported details.

In [4]:
results_dir = project_root / "reports" / "caption_baseline_v1"
results_dir.mkdir(parents=True, exist_ok=True)

prediction_document = {
    "schema_version": 1,
    "checkpoint_sha256": file_sha256(checkpoint_path),
    "selected_epoch": selected_checkpoint["epoch"],
    "split_sha256": selected_checkpoint["split_sha256"],
    "vocabulary_sha256": selected_checkpoint["vocabulary_sha256"],
    "image_preprocessing_sha256": (
        selected_checkpoint["image_preprocessing_sha256"]
    ),
    "development_feature_sha256": (
        development_metadata["feature_sha256"]
    ),
    "decoding": {
        "method": "greedy",
        "maximum_new_tokens": 39,
        "batch_size": generation_batch_size,
    },
    "mismatch": {
        "method": "Seeded permutation connected into a single cycle",
        "seed": 42,
        "self_matches": 0,
    },
    "torch_version": str(torch.__version__),
    "device": str(device),
    "summary": {
        "images": image_count,
        "captions_changed_under_mismatch": changed_captions,
        "correct_feature_eos_count": correct_eos,
        "mismatched_feature_eos_count": wrong_eos,
    },
    "predictions": development_predictions,
}

prediction_path = results_dir / "development_predictions_v1.json"
prediction_bytes = (
    json.dumps(prediction_document, sort_keys=True, indent=2) + "\n"
).encode("utf-8")

if prediction_path.exists():
    if prediction_path.read_bytes() != prediction_bytes:
        raise FileExistsError(
            "Saved predictions differ; preserve and review before replacing."
        )
else:
    temporary_path = prediction_path.with_suffix(".json.part")
    temporary_path.write_bytes(prediction_bytes)
    temporary_path.replace(prediction_path)

print("Saved:", prediction_path.relative_to(project_root))
print("Prediction records:", len(development_predictions))
print("SHA-256:", file_sha256(prediction_path))

Saved: reports/caption_baseline_v1/development_predictions_v1.json
Prediction records: 200
SHA-256: aefa1a61ff5c0fb9c248736b05438c9eab70a9e8d2016262f383322a2e527be2


## Nearest-Training-Image Caption Baseline

Each image is represented by the mean of its 49 frozen ResNet18 feature vectors. Cosine similarity identifies the closest training image for each development image.

The retrieved caption is the matched training image's lowest-ID human caption. Development references do not influence image retrieval or caption selection.

This baseline tests whether caption generation improves on retrieving a caption from a visually similar training image. It uses the same frozen encoder features as the generator.

In [5]:
from zipfile import ZipFile

training_metadata = json.loads(
    (cache_dir / "train_resnet18_v1.json").read_text(encoding="utf-8")
)

expected_training_values = {
    "split": "train",
    "image_ids": sorted(splits["image_ids"]["train"]),
    "split_manifest_sha256": selected_checkpoint["split_sha256"],
    "image_preprocessing_sha256": (
        selected_checkpoint["image_preprocessing_sha256"]
    ),
    "encoder": "resnet18",
    "encoder_weights": "IMAGENET1K_V1",
    "feature_sha256": selected_checkpoint["feature_sha256"]["train"],
}
for key, expected_value in expected_training_values.items():
    if training_metadata.get(key) != expected_value:
        raise ValueError(f"Training cache metadata differs: {key}")

training_feature_path = cache_dir / training_metadata["feature_file"]
if file_sha256(training_feature_path) != training_metadata["feature_sha256"]:
    raise ValueError("Training feature checksum differs.")

training_features = np.load(
    training_feature_path,
    mmap_mode="r",
    allow_pickle=False,
)
if (
    training_features.shape
    != (len(training_metadata["image_ids"]), 49, 512)
    or training_features.dtype != np.float32
):
    raise ValueError("Training feature shape or dtype differs.")


def pooled_unit_vectors(features):
    """Average spatial features and normalize each image vector.

    Parameters
    ----------
    features : numpy.ndarray
        Feature array of shape (N, 49, 512), optionally memory-mapped.

    Returns
    -------
    numpy.ndarray
        Float32 array of shape (N, 512), with unit-length rows.
    """
    pooled = features.mean(axis=1, dtype=np.float32)
    norms = np.linalg.norm(pooled, axis=1, keepdims=True)

    if not np.isfinite(pooled).all() or np.any(norms == 0):
        raise ValueError("Cannot normalize invalid or zero feature vectors.")

    return pooled / norms


training_vectors = pooled_unit_vectors(training_features)
development_vectors = pooled_unit_vectors(development_features)

# Dot products of unit vectors equal cosine similarities.
similarities = development_vectors @ training_vectors.T
nearest_rows = similarities.argmax(axis=1)
nearest_image_ids = [
    training_metadata["image_ids"][int(row)]
    for row in nearest_rows
]

# Verify and read the original annotation member.
source_provenance = json.loads(
    (manifest_dir / "train2017_candidates.provenance.json").read_text(
        encoding="utf-8"
    )
)
annotation_archive = (
    project_root / "feasibility" / source_provenance["source_archive"]
)

with ZipFile(annotation_archive) as archive:
    annotation_bytes = archive.read(
        source_provenance["annotation_member"]
    )

if (
    hashlib.sha256(annotation_bytes).hexdigest()
    != source_provenance["annotation_sha256"]
):
    raise ValueError("Annotation source checksum differs.")

annotations = json.loads(annotation_bytes)["annotations"]
del annotation_bytes

development_references = {
    image_id: [] for image_id in development_image_ids
}
nearest_id_set = set(nearest_image_ids)
retrieval_captions = {}

# Ascending caption IDs make the retrieval-caption choice deterministic.
for annotation in sorted(annotations, key=lambda record: record["id"]):
    image_id = annotation["image_id"]

    if image_id in development_references:
        development_references[image_id].append({
            "caption_id": annotation["id"],
            "caption": annotation["caption"],
        })

    if image_id in nearest_id_set and image_id not in retrieval_captions:
        retrieval_captions[image_id] = {
            "caption_id": annotation["id"],
            "caption": annotation["caption"],
        }

del annotations

if any(not refs for refs in development_references.values()):
    raise ValueError("Development reference coverage is incomplete.")
if set(retrieval_captions) != nearest_id_set:
    raise ValueError("A retrieved training image has no caption.")

retrieval_records = []
for row, image_id in enumerate(development_image_ids):
    neighbor_id = nearest_image_ids[row]
    retrieval_records.append({
        "image_id": image_id,
        "training_image_id": neighbor_id,
        "cosine_similarity": float(similarities[row, nearest_rows[row]]),
        "training_caption_id": retrieval_captions[neighbor_id]["caption_id"],
        "retrieved_caption": retrieval_captions[neighbor_id]["caption"],
        "references": development_references[image_id],
    })

retrieval_document = {
    "schema_version": 1,
    "split_sha256": selected_checkpoint["split_sha256"],
    "training_feature_sha256": training_metadata["feature_sha256"],
    "development_feature_sha256": development_metadata["feature_sha256"],
    "annotation_sha256": source_provenance["annotation_sha256"],
    "method": "Mean spatial features; cosine nearest training image",
    "caption_selection": "Lowest annotation caption_id for matched image",
    "records": retrieval_records,
}

retrieval_path = results_dir / "development_retrieval_baseline_v1.json"
retrieval_bytes = (
    json.dumps(retrieval_document, sort_keys=True, indent=2) + "\n"
).encode("utf-8")

if retrieval_path.exists():
    if retrieval_path.read_bytes() != retrieval_bytes:
        raise FileExistsError("Existing retrieval results differ.")
else:
    temporary_path = retrieval_path.with_suffix(".json.part")
    temporary_path.write_bytes(retrieval_bytes)
    temporary_path.replace(retrieval_path)

print("Development images with references:", len(development_references))
print("Retrieval predictions:", len(retrieval_records))
print(
    "Retrieved images all belong to training:",
    set(nearest_image_ids).issubset(set(splits["image_ids"]["train"])),
)
print("Saved:", retrieval_path.relative_to(project_root))

for record in retrieval_records[:6]:
    print(
        f"\nDevelopment image {record['image_id']} "
        f"→ training image {record['training_image_id']}"
    )
    print("Retrieved:", record["retrieved_caption"])

del similarities, training_vectors, development_vectors

Development images with references: 200
Retrieval predictions: 200
Retrieved images all belong to training: True
Saved: reports/caption_baseline_v1/development_retrieval_baseline_v1.json

Development image 1323 → training image 59361
Retrieved: a fence that has a bunch of giraffe in it

Development image 2471 → training image 311167
Retrieved: A sleepy calico cat relaxing in a sink. 

Development image 7934 → training image 289187
Retrieved: A close up picture of a granny smith apple

Development image 8218 → training image 439930
Retrieved: A large clock tower towering over the top of a building.

Development image 13124 → training image 139878
Retrieved: one skier is skiing and some trees are in the background

Development image 15554 → training image 574562
Retrieved: A woman walking down the street with an umbrella.


In [7]:
import nltk
from nltk.translate.bleu_score import corpus_bleu

print("NLTK:", nltk.__version__)

NLTK: 3.10.4


## Reference-Overlap Evaluation

Corpus BLEU-1 measures unigram overlap with a brevity penalty. Corpus BLEU-4 combines overlap for one- through four-token sequences with a brevity penalty.

All three conditions use the same 200 development images, human references, and frozen caption tokenizer. References retain their full text and are not mapped through the training vocabulary. No smoothing is applied. Scores are reported on a 0–100 scale.

BLEU measures reference overlap, not visual factuality. The mismatched feature condition tests the benefit of correct image input. The retrieval baseline provides a comparison with an existing training caption.

In [8]:
import re

caption_preprocessing = json.loads(
    (manifest_dir / "caption_preprocessing_v1.json").read_text(
        encoding="utf-8"
    )
)

if (
    caption_preprocessing["split_manifest_sha256"]
    != selected_checkpoint["split_sha256"]
    or caption_preprocessing["vocabulary_sha256"]
    != selected_checkpoint["vocabulary_sha256"]
):
    raise ValueError("Caption preprocessing provenance differs.")

evaluation_token_pattern = re.compile(
    caption_preprocessing["tokenizer"]["pattern"]
)


def evaluation_tokens(text):
    """Tokenize caption text using the frozen preprocessing rules.

    Parameters
    ----------
    text : str
        Generated, retrieved, or reference caption.

    Returns
    -------
    list[str]
        Tokens without vocabulary mapping or truncation.
    """
    if caption_preprocessing["tokenizer"]["lowercase"]:
        text = text.lower()
    return evaluation_token_pattern.findall(text)


prediction_lookup = {
    record["image_id"]: record
    for record in development_predictions
}
retrieval_lookup = {
    record["image_id"]: record
    for record in retrieval_records
}

expected_ids = set(development_image_ids)
if (
    len(prediction_lookup) != len(development_predictions)
    or len(retrieval_lookup) != len(retrieval_records)
    or set(prediction_lookup) != expected_ids
    or set(retrieval_lookup) != expected_ids
):
    raise ValueError("Evaluation image coverage or uniqueness differs.")

reference_tokens = [
    [
        evaluation_tokens(reference["caption"])
        for reference in retrieval_lookup[image_id]["references"]
    ]
    for image_id in development_image_ids
]

condition_tokens = {
    "correct_features": [
        evaluation_tokens(
            prediction_lookup[image_id]["correct_feature_caption"]
        )
        for image_id in development_image_ids
    ],
    "mismatched_features": [
        evaluation_tokens(
            prediction_lookup[image_id]["mismatched_feature_caption"]
        )
        for image_id in development_image_ids
    ],
    "retrieval_baseline": [
        evaluation_tokens(
            retrieval_lookup[image_id]["retrieved_caption"]
        )
        for image_id in development_image_ids
    ],
}

bleu_results = {}
for condition, hypotheses in condition_tokens.items():
    bleu_results[condition] = {
        "bleu_1": 100 * corpus_bleu(
            reference_tokens,
            hypotheses,
            weights=(1.0,),
            smoothing_function=None,
            auto_reweigh=False,
        ),
        "bleu_4": 100 * corpus_bleu(
            reference_tokens,
            hypotheses,
            weights=(0.25, 0.25, 0.25, 0.25),
            smoothing_function=None,
            auto_reweigh=False,
        ),
    }

print(f"{'Condition':<23} {'BLEU-1':>9} {'BLEU-4':>9}")
for condition, scores in bleu_results.items():
    print(
        f"{condition:<23} "
        f"{scores['bleu_1']:>9.2f} "
        f"{scores['bleu_4']:>9.2f}"
    )

for baseline in ("mismatched_features", "retrieval_baseline"):
    print(f"\nCorrect features minus {baseline}:")
    for metric in ("bleu_1", "bleu_4"):
        difference = (
            bleu_results["correct_features"][metric]
            - bleu_results[baseline][metric]
        )
        print(f"{metric}: {difference:+.2f} points")

Condition                  BLEU-1    BLEU-4
correct_features            63.01     21.33
mismatched_features         43.08      4.22
retrieval_baseline          51.88     10.56

Correct features minus mismatched_features:
bleu_1: +19.93 points
bleu_4: +17.11 points

Correct features minus retrieval_baseline:
bleu_1: +11.13 points
bleu_4: +10.77 points


## Paired Human Grounding Review

All 200 development images are reviewed against generated and retrieved captions. Caption identities are concealed behind fixed randomized A/B labels during scoring.

Each caption receives a main-subject/action judgment and separate flags for materially unsupported objects, actions, attributes, and relationships. Omitted secondary details do not count as unsupported claims.

Overall pass requires a correct main subject/action and no material unsupported claim. Evaluator identity and notes are retained. Ratings are saved after each completed pair.

In [9]:
import html
from datetime import datetime, timezone

import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image, ImageOps

inventory = json.loads(
    (manifest_dir / "train2017_candidates.json").read_text(
        encoding="utf-8"
    )
)
review_image_lookup = {
    record["image_id"]: record for record in inventory
}

review_protocol_path = results_dir / "human_review_protocol_v1.json"
review_ratings_path = results_dir / "human_review_ratings_v1.json"

# Assign A/B independently for each image, without using caption quality.
review_rng = np.random.default_rng(31415)
review_pairs = []

for image_id in development_image_ids:
    candidates = [
        (
            "generator",
            prediction_lookup[image_id]["correct_feature_caption"],
        ),
        (
            "retrieval",
            retrieval_lookup[image_id]["retrieved_caption"],
        ),
    ]
    if review_rng.integers(2):
        candidates.reverse()

    review_pairs.append({
        "image_id": image_id,
        "A": {
            "condition": candidates[0][0],
            "caption": candidates[0][1],
        },
        "B": {
            "condition": candidates[1][0],
            "caption": candidates[1][1],
        },
    })

review_protocol = {
    "schema_version": 1,
    "assignment_seed": 31415,
    "prediction_sha256": file_sha256(prediction_path),
    "retrieval_sha256": file_sha256(retrieval_path),
    "pairs": review_pairs,
}
protocol_bytes = (
    json.dumps(review_protocol, sort_keys=True, indent=2) + "\n"
).encode("utf-8")

if review_protocol_path.exists():
    if review_protocol_path.read_bytes() != protocol_bytes:
        raise ValueError("Existing review assignments differ.")
else:
    temporary_path = review_protocol_path.with_suffix(".json.part")
    temporary_path.write_bytes(protocol_bytes)
    temporary_path.replace(review_protocol_path)

review_protocol_sha256 = file_sha256(review_protocol_path)

if review_ratings_path.exists():
    review_ratings = json.loads(
        review_ratings_path.read_text(encoding="utf-8")
    )
    if review_ratings["protocol_sha256"] != review_protocol_sha256:
        raise ValueError("Saved ratings belong to another review protocol.")
else:
    review_ratings = {
        "schema_version": 1,
        "protocol_sha256": review_protocol_sha256,
        "ratings": {},
    }

print("Review pairs:", len(review_pairs))
print("Previously completed:", len(review_ratings["ratings"]))

Review pairs: 200
Previously completed: 0


In [10]:
rating_fields = [
    ("main_correct", "Main subject/action correct"),
    ("object_error", "Unsupported object"),
    ("action_error", "Unsupported action"),
    ("attribute_error", "Unsupported attribute"),
    ("relationship_error", "Unsupported relationship"),
]

evaluator_input = widgets.Text(
    description="Evaluator:",
    placeholder="Enter your full name",
    layout=widgets.Layout(width="500px"),
)
review_position = widgets.BoundedIntText(
    value=1,
    min=1,
    max=len(review_pairs),
    description="Pair:",
)
load_button = widgets.Button(description="Load pair")
save_button = widgets.Button(
    description="Save & next", button_style="success"
)
review_output = widgets.Output()
review_status = widgets.HTML()
review_notes = widgets.Textarea(
    description="Notes:",
    placeholder="Explain material errors or uncertainty.",
    layout=widgets.Layout(width="95%", height="80px"),
)

rating_controls = {
    label: {
        key: widgets.Dropdown(
            options=[("Select", None), ("Yes", 1), ("No", 0)],
            value=None,
            description=description,
            style={"description_width": "190px"},
            layout=widgets.Layout(width="310px"),
        )
        for key, description in rating_fields
    }
    for label in ("A", "B")
}


def load_review_pair(_=None):
    """Display the selected pair and restore any saved ratings.

    Parameters
    ----------
    _ : object, optional
        Button callback event; ignored.

    Returns
    -------
    None
        Updates the image, captions, and rating controls.
    """
    pair = review_pairs[review_position.value - 1]
    saved = review_ratings["ratings"].get(str(pair["image_id"]), {})

    for label in ("A", "B"):
        for key, _description in rating_fields:
            rating_controls[label][key].value = (
                saved.get(label, {}).get(key)
            )

    review_notes.value = saved.get("notes", "")
    if saved.get("evaluator"):
        evaluator_input.value = saved["evaluator"]

    image_path = (
        project_root / "data" / "raw" / "coco" / "train2017"
        / review_image_lookup[pair["image_id"]]["file_name"]
    )
    with Image.open(image_path) as source:
        preview = ImageOps.exif_transpose(source).convert("RGB")
        preview.thumbnail((640, 420))

    with review_output:
        clear_output(wait=True)
        print(
            f"Pair {review_position.value}/{len(review_pairs)} "
            f"| Image {pair['image_id']}"
        )
        display(preview)

        panels = []
        for label in ("A", "B"):
            panels.append(widgets.VBox([
                widgets.HTML(
                    f"<b>Caption {label}</b><p>"
                    f"{html.escape(pair[label]['caption'])}</p>"
                ),
                *rating_controls[label].values(),
            ], layout=widgets.Layout(width="49%")))
        display(widgets.HBox(panels))


def save_review_pair(_):
    """Validate and atomically save the displayed pair's human ratings.

    Parameters
    ----------
    _ : object
        Button callback event; ignored.

    Returns
    -------
    None
        Saves ratings and advances to the next pair.
    """
    evaluator = evaluator_input.value.strip()
    if not evaluator:
        review_status.value = "<b>Enter the evaluator name first.</b>"
        return

    ratings = {
        label: {
            key: control.value
            for key, control in rating_controls[label].items()
        }
        for label in ("A", "B")
    }
    if any(
        value is None
        for caption_ratings in ratings.values()
        for value in caption_ratings.values()
    ):
        review_status.value = "<b>Complete all ten judgments first.</b>"
        return

    pair = review_pairs[review_position.value - 1]
    review_ratings["ratings"][str(pair["image_id"])] = {
        "evaluator": evaluator,
        "reviewed_at_utc": datetime.now(timezone.utc).isoformat(),
        "A": ratings["A"],
        "B": ratings["B"],
        "notes": review_notes.value.strip(),
    }

    temporary_path = review_ratings_path.with_suffix(".json.part")
    temporary_path.write_text(
        json.dumps(review_ratings, sort_keys=True, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary_path.replace(review_ratings_path)

    completed = len(review_ratings["ratings"])
    review_status.value = (
        f"<b>Saved. Completed {completed}/{len(review_pairs)} pairs.</b>"
    )

    if review_position.value < len(review_pairs):
        review_position.value += 1
        load_review_pair()


load_button.on_click(load_review_pair)
save_button.on_click(save_review_pair)

# Resume at the first unreviewed image.
for index, pair in enumerate(review_pairs, start=1):
    if str(pair["image_id"]) not in review_ratings["ratings"]:
        review_position.value = index
        break

display(
    evaluator_input,
    widgets.HBox([review_position, load_button]),
    review_output,
    review_notes,
    save_button,
    review_status,
)
load_review_pair()

Text(value='', description='Evaluator:', layout=Layout(width='500px'), placeholder='Enter your full name')

Output()

Textarea(value='', description='Notes:', layout=Layout(height='80px', width='95%'), placeholder='Explain mater…

Button(button_style='success', description='Save & next', style=ButtonStyle())

HTML(value='')

In [11]:
# import io
# import json
# from pathlib import Path
# from zipfile import ZipFile, ZIP_DEFLATED

# from PIL import Image, ImageOps

# root = Path.cwd().resolve()
# protocol_path = (
#     root / "reports" / "caption_baseline_v1"
#     / "human_review_protocol_v1.json"
# )
# protocol = json.loads(protocol_path.read_text(encoding="utf-8"))

# inventory = json.loads(
#     (root / "manifests" / "train2017_candidates.json").read_text(
#         encoding="utf-8"
#     )
# )
# image_lookup = {record["image_id"]: record for record in inventory}

# # Include only image IDs and caption text; preserve the original locally.
# upload_pairs = [
#     {
#         "image_id": pair["image_id"],
#         "A": {"caption": pair["A"]["caption"]},
#         "B": {"caption": pair["B"]["caption"]},
#     }
#     for pair in protocol["pairs"]
# ]

# zip_path = root / "data" / "processed" / "caption_review_upload.zip"
# zip_path.parent.mkdir(parents=True, exist_ok=True)
# temporary_path = zip_path.with_suffix(".zip.part")

# with ZipFile(temporary_path, "w", compression=ZIP_DEFLATED) as package:
#     package.writestr(
#         "review_captions.json",
#         json.dumps({"pairs": upload_pairs}, indent=2),
#     )

#     for pair in upload_pairs:
#         image_id = pair["image_id"]
#         image_path = (
#             root / "data" / "raw" / "coco" / "train2017"
#             / image_lookup[image_id]["file_name"]
#         )
#         with Image.open(image_path) as source:
#             preview = ImageOps.exif_transpose(source).convert("RGB")
#             preview.thumbnail((1024, 1024))
#             buffer = io.BytesIO()
#             preview.save(buffer, format="JPEG", quality=95)

#         package.writestr(
#             f"images/{image_id}.jpg",
#             buffer.getvalue(),
#         )

# temporary_path.replace(zip_path)

# print("Review pairs:", len(upload_pairs))
# print("Upload:", zip_path)
# print("Size (MiB):", round(zip_path.stat().st_size / 1024**2, 2))

Review pairs: 200
Upload: /Users/ashlabs/Projects/Masters in AI/github/image-inspired-story-creator/data/processed/caption_review_upload.zip
Size (MiB): 24.78


## Confirmed Grounding Results

The completed review covers 200 development images and 400 captions. Ratings are read from caption_review_confirmed.json. The original A/B protocol identifies each caption's condition.

A caption passes when its main subject/action is correct and none of the four material-error flags is set. The major object/action error rate counts captions with either error, without double-counting.

The paired comparison resamples image pairs together, using 10,000 bootstrap samples and seed 42.

In [12]:
confirmed_path = results_dir / "caption_review_confirmed.json"
protocol_path = results_dir / "human_review_protocol_v1.json"
summary_path = results_dir / "human_review_summary_v1.json"

confirmed_review = json.loads(
    confirmed_path.read_text(encoding="utf-8")
)
review_protocol = json.loads(
    protocol_path.read_text(encoding="utf-8")
)
saved_summary = json.loads(
    summary_path.read_text(encoding="utf-8")
)

if file_sha256(confirmed_path) != saved_summary["ratings_sha256"]:
    raise ValueError("Confirmed ratings differ from the saved analysis.")
if file_sha256(protocol_path) != saved_summary["protocol_sha256"]:
    raise ValueError("Review protocol differs from the saved analysis.")

protocol_lookup = {
    pair["image_id"]: pair for pair in review_protocol["pairs"]
}
confirmed_pairs = confirmed_review["human_review"]["pairs"]
proposal_lookup = {
    pair["image_id"]: pair
    for pair in confirmed_review["proposed_pairs"]
}
confirmed_lookup = {
    pair["image_id"]: pair for pair in confirmed_pairs
}

expected_ids = set(development_image_ids)
if (
    len(confirmed_pairs) != len(expected_ids)
    or set(confirmed_lookup) != expected_ids
    or len(review_protocol["pairs"]) != len(expected_ids)
    or set(protocol_lookup) != expected_ids
    or len(confirmed_review["proposed_pairs"]) != len(expected_ids)
    or set(proposal_lookup) != expected_ids
):
    raise ValueError("Review coverage or image uniqueness differs.")

rating_fields = [
    "main_correct",
    "object_error",
    "action_error",
    "attribute_error",
    "relationship_error",
]
scored_review = {"generator": [], "retrieval": []}

for image_id in sorted(expected_ids):
    human_pair = confirmed_lookup[image_id]
    protocol_pair = protocol_lookup[image_id]

    if (
        human_pair["human_reviewed"] is not True
        or not human_pair["evaluator"].strip()
        or not human_pair["reviewed_at_utc"]
    ):
        raise ValueError(f"Unconfirmed or incomplete review: {image_id}")

    if {
        protocol_pair[label]["condition"] for label in ("A", "B")
    } != {"generator", "retrieval"}:
        raise ValueError(f"Invalid condition mapping: {image_id}")

    for label in ("A", "B"):
        if (
            protocol_pair[label]["caption"]
            != proposal_lookup[image_id][label]["caption"]
        ):
            raise ValueError(f"Caption assignment differs: {image_id}")

        ratings = human_pair[label]["ratings"]
        if (
            set(ratings) != set(rating_fields)
            or any(
                type(value) is not int or value not in (0, 1)
                for value in ratings.values()
            )
        ):
            raise ValueError(f"Invalid ratings: {image_id}, {label}")

        condition = protocol_pair[label]["condition"]
        scored_review[condition].append({
            "image_id": image_id,
            **ratings,
            "overall_pass": int(
                ratings["main_correct"] == 1
                and all(ratings[key] == 0 for key in rating_fields[1:])
            ),
            "major_object_or_action": int(
                ratings["object_error"] == 1
                or ratings["action_error"] == 1
            ),
        })

metrics = rating_fields + ["overall_pass", "major_object_or_action"]

print(f"{'Measure':<27} {'Generator':>12} {'Retrieval':>12}")
for metric in metrics:
    percentages = []
    for condition in ("generator", "retrieval"):
        count = sum(row[metric] for row in scored_review[condition])

        # Check the independently calculated result against the saved report.
        if count != saved_summary["methods"][condition][metric]["count"]:
            raise ValueError(f"Saved count differs: {condition}, {metric}")

        percentages.append(100 * count / len(expected_ids))

    print(
        f"{metric:<27} "
        f"{percentages[0]:>11.1f}% "
        f"{percentages[1]:>11.1f}%"
    )

generator_passes = np.array([
    row["overall_pass"] for row in scored_review["generator"]
])
retrieval_passes = np.array([
    row["overall_pass"] for row in scored_review["retrieval"]
])

# Identical image ordering preserves pairing during resampling.
paired_differences = generator_passes - retrieval_passes
bootstrap_rng = np.random.default_rng(42)
bootstrap_indices = bootstrap_rng.integers(
    0, len(expected_ids), size=(10_000, len(expected_ids))
)
bootstrap_differences = (
    paired_differences[bootstrap_indices].mean(axis=1) * 100
)
interval = np.percentile(bootstrap_differences, [2.5, 97.5])
difference = paired_differences.mean() * 100

if not np.isclose(
    difference,
    saved_summary["paired_comparison"]["difference_percentage_points"],
):
    raise ValueError("Saved paired difference differs.")
if not np.allclose(
    interval,
    saved_summary["paired_comparison"][
        "paired_bootstrap_95_percentage_points"
    ],
):
    raise ValueError("Saved bootstrap interval differs.")

print(f"\nPass-rate difference: {difference:+.1f} percentage points")
print(
    f"Paired 95% bootstrap interval: "
    f"[{interval[0]:+.1f}, {interval[1]:+.1f}]"
)
print("Saved analysis reproduced: verified")

Measure                        Generator    Retrieval
main_correct                       72.5%        64.0%
object_error                       53.0%        53.0%
action_error                       12.5%        23.5%
attribute_error                    18.0%        25.0%
relationship_error                 30.0%        35.5%
overall_pass                       28.5%        24.0%
major_object_or_action             58.5%        59.5%

Pass-rate difference: +4.5 percentage points
Paired 95% bootstrap interval: [-3.0, +12.0]
Saved analysis reproduced: verified


### Baseline Evaluation Conclusion

The generator achieved a 28.5% overall pass rate, compared with 24.0% for nearest-image caption retrieval. Its 4.5-percentage-point advantage had a paired 95% bootstrap interval of −3.0 to +12.0 points, making
the comparison inconclusive.

Major unsupported objects or actions occurred in 58.5% of generated captions. The baseline did not meet the predefined caption-quality targets.

Although correct image features improved BLEU and generation exceeded retrieval on reference overlap, these improvements did not establish reliable visual grounding. The baseline checkpoint remains a comparison point for subsequent controlled changes. The final test set remains unused.